In [0]:
%pip install torch transformers ipywidgets requests

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import logging
import mlflow
import requests
import shutil
from datetime import datetime
from PIL import Image, ImageDraw, ImageFont
from transformers import pipeline

# ================= CONFIGURAÇÕES GERAIS =================
CATALOG = "AI"
SCHEMA = "AISTROS"
VOLUME = "IMAGES"
BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Garante a estrutura de pastas
os.makedirs(os.path.join(BASE_PATH, "processed"), exist_ok=True)
os.makedirs(os.path.join(BASE_PATH, "logs"), exist_ok=True)

# Modelo VQA
HF_MODEL_NAME = "dandelin/vilt-b32-finetuned-vqa" 
TASK = "visual-question-answering"

print(f"✅ Configuração concluída. Diretório raiz: {BASE_PATH}")

In [0]:
# ================= FUNÇÕES AUXILIARES =================

def setup_logger(log_path):
    """Configura logs para arquivo e console"""
    logger = logging.getLogger("AstroLogger")
    if logger.hasHandlers(): logger.handlers.clear()
    logger.setLevel(logging.INFO)
    
    formatter = logging.Formatter('%(asctime)s - %(message)s')
    fh = logging.FileHandler(log_path, mode='w'); fh.setFormatter(formatter)
    ch = logging.StreamHandler(); ch.setFormatter(formatter)
    
    logger.addHandler(fh); logger.addHandler(ch)
    return logger

def download_image(url):
    """Baixa imagem da URL e salva no Volume. Retorna o nome do arquivo."""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        # Cria nome único baseado na hora
        filename = f"download_{datetime.now().strftime('%H%M%S')}.jpg"
        save_path = os.path.join(BASE_PATH, filename)
        
        print(f"⬇️ Baixando: {url}...")
        response = requests.get(url, headers=headers, stream=True, timeout=15)
        
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                response.raw.decode_content = True
                shutil.copyfileobj(response.raw, f)
            print(f"✅ Download salvo em: {save_path}")
            return filename
        else:
            print(f"❌ Erro HTTP {response.status_code}")
            return None
    except Exception as e:
        print(f"❌ Erro no download: {e}")
        return None

def processar_astro_imagem(nome_arquivo):
    """Pipeline principal: Carrega, Analisa (IA), Anota e Salva"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name = os.path.splitext(nome_arquivo)[0]
    
    # Definição dos caminhos
    path_in = os.path.join(BASE_PATH, nome_arquivo)
    path_out = os.path.join(BASE_PATH, "processed", f"proc_{base_name}_{timestamp}.png")
    path_log = os.path.join(BASE_PATH, "logs", f"log_{base_name}_{timestamp}.txt")
    
    # Inicia Logger e MLflow
    logger = setup_logger(path_log)
    current_user = spark.sql("SELECT current_user()").first()[0]
    mlflow.set_experiment(f"/Users/{current_user}/Astro_Recognition")
    
    with mlflow.start_run(run_name=f"VQA_{base_name}"):
        try:
            logger.info(f"--- PROCESSANDO: {nome_arquivo} ---")
            
            if not os.path.exists(path_in):
                raise FileNotFoundError(f"Arquivo não encontrado no volume: {path_in}")
                
            # 1. Carregar Imagem
            image = Image.open(path_in).convert("RGB")
            mlflow.log_image(image, "original.png")
            
            # 2. Inferência (IA)
            logger.info(f"Carregando modelo: {HF_MODEL_NAME}...")
            pipe = pipeline(TASK, model=HF_MODEL_NAME)
            
            questions = [
                "Describe the image in detail.",
                "Is this a photo of stars or a galaxy?",
                "Are there any geometric patterns?"
            ]
            
            results_txt = []
            logger.info("--- INÍCIO VQA ---")
            for q in questions:
                ans = pipe(image, q, top_k=1)[0]
                texto_res = f"P: {q}\nR: {ans['answer']} (Conf: {ans['score']:.2f})"
                logger.info(texto_res.replace('\n', ' -> '))
                results_txt.append(texto_res)
                mlflow.log_metric(f"score_{q.split()[0]}", ans['score'])
            
            # 3. Anotação na Imagem
            draw = ImageDraw.Draw(image)
            # Tenta fonte padrão se não tiver arial/dejavu
            font = ImageFont.load_default()
            
            draw.text((10, 10), "\n".join(results_txt), fill="white", font=font)
            
            # 4. Salvar Resultados
            image.save(path_out)
            mlflow.log_image(image, "processed_annotated.png")
            logger.info(f"--- SUCESSO: Imagem salva em {path_out} ---")
            
            # Finaliza log para upload
            handlers = logger.handlers[:]
            for h in handlers: h.close(); logger.removeHandler(h)
            mlflow.log_artifact(path_log, "logs")
            
            return path_out
            
        except Exception as e:
            logger.error(f"FALHA CRÍTICA: {e}")
            raise e

In [0]:
# ================= CONTROLE DE EXECUÇÃO =================

# ESCOLHA O MODO: "URL" ou "VOLUME"
MODO = "VOLUME" 

# DIGITE A URL (se MODO="URL") OU O NOME DO ARQUIVO (se MODO="VOLUME")
# Exemplo URL: "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c2/Orion_constellation_map.svg/800px-Orion_constellation_map.svg.png"
# Exemplo Volume: "minha_foto.png"
ENTRADA = "orion.jpg"


# --- Lógica de Decisão ---
print(f"🚀 Iniciando execução no modo: {MODO}")
arquivo_para_processar = None

if MODO == "URL":
    # Baixa a imagem e pega o nome gerado
    arquivo_para_processar = download_image(ENTRADA)
    
elif MODO == "VOLUME":
    # Verifica se existe no volume
    caminho_completo = os.path.join(BASE_PATH, ENTRADA)
    if os.path.exists(caminho_completo):
        print(f"✅ Arquivo localizado no volume: {ENTRADA}")
        arquivo_para_processar = ENTRADA
    else:
        print(f"❌ Arquivo não encontrado: {caminho_completo}")

# --- Executa o Pipeline se tivermos um arquivo válido ---
if arquivo_para_processar:
    try:
        caminho_final = processar_astro_imagem(arquivo_para_processar)
        print(f"\n✨ Processo finalizado com sucesso!")
        print(f"📂 Resultado salvo em: {caminho_final}")
    except Exception as e:
        print(f"❌ Ocorreu um erro durante o processamento: {e}")
else:
    print("⚠️ Nenhuma imagem válida para processar.")